# Lab: Fleet Reliability from Real LLM-Cluster Data (AcmeTrace)

**Goal:** derive failure rates, MTBF, and a goodput waterfall from *real* LLM-development cluster traces — then test the assumptions in the [pod-scale training answer](https://sys-design-primer-cvw.pages.dev/google-interview/6-answer-pod-training/) (fleet MTBF ≈ 2.4 h at ~18K chips, ~5-year per-chip MTBF prior) against data.

**Data:** Shanghai AI Lab's Acme traces (NSDI '24, ["Characterization of LLM Development in the Datacenter"](https://arxiv.org/abs/2403.07648)) — 6 months (Mar–Aug 2023), two A100 clusters: **Seren** (pretraining-heavy) and **Kalos** (mixed dev). 880K job records: submit/start/end, GPU count, terminal state — including an explicit **`NODE_FAIL`** state, which is the direct infrastructure-failure signal (generic `FAILED` conflates user bugs, OOM, and infra).

**Caveats up front:** job records, not hardware logs — one node failure kills one *job* record even if several nodes died, and auto-restarted jobs re-enter as new records; the two clusters have slightly different schemas; chip-hours are A100-hours, not TPU-hours. The point is the *method* and the order of magnitude.

In [ ]:
import pandas as pd, urllib.request, os
BASE = "https://raw.githubusercontent.com/InternLM/AcmeTrace/master/data/job_trace"  # note: master, not main
frames = []
for c in ("seren", "kalos"):
    f = f"trace_{c}.csv"
    if not os.path.exists(f):
        print(f"downloading {f} (seren is ~99 MB)…")
        urllib.request.urlretrieve(f"{BASE}/{f}", f)
    d = pd.read_csv(f)
    # schemas differ between clusters (seren has no fail_time) — parse what exists
    for col in ("submit_time", "start_time", "end_time", "fail_time"):
        if col in d.columns:
            d[col] = pd.to_datetime(d[col], errors="coerce", utc=True)
    d["cluster"] = c
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print(df.shape)
df.state.value_counts()

Expected: ~880K rows; states COMPLETED (~571K), FAILED (~204K), CANCELLED (~62K), PREEMPTED (~44K), **NODE_FAIL (199)**, TIMEOUT (75). Note the asymmetry you'll use later: NODE_FAIL is rare by *count* — the question is what it costs in GPU-hours.

## Part 1 — What does a real cluster's job mix look like?
**Commit a guess before running:** what fraction of *jobs* are ≥64 GPUs? What fraction of *GPU-hours* do they consume?

In [ ]:
gpu_jobs = df[df.gpu_num > 0].copy()
gpu_jobs["gpu_hours"] = gpu_jobs.gpu_num * gpu_jobs.duration / 3600
bins = [0, 1, 8, 32, 64, 128, 512, 100000]
labels = ["1", "2-8", "9-32", "33-64", "65-128", "129-512", "513+"]
gpu_jobs["size_tier"] = pd.cut(gpu_jobs.gpu_num, bins=bins, labels=labels)
summary = gpu_jobs.groupby("size_tier", observed=True).agg(
    jobs=("job_id", "count"), gpu_hours=("gpu_hours", "sum"))
summary["job_share_%"] = (100 * summary.jobs / summary.jobs.sum()).round(2)
summary["gpu_hour_share_%"] = (100 * summary.gpu_hours / summary.gpu_hours.sum()).round(2)
summary

The classic result: large jobs are a sliver of job *count* and a dominant share of GPU-*hours* — which is why reliability engineering targets the big-job tail.

## Part 2 — Terminal states by scale
Does the probability of a bad ending grow with GPU count? Watch both `FAILED` and `NODE_FAIL` columns — and notice which tiers the NODE_FAIL GPU-hours concentrate in.

In [ ]:
by_tier = gpu_jobs.groupby("size_tier", observed=True).apply(
    lambda g: pd.Series({
        "jobs": len(g),
        "failed_%": 100 * (g.state == "FAILED").mean(),
        "node_fail_%": 100 * (g.state == "NODE_FAIL").mean(),
        "completed_%": 100 * (g.state == "COMPLETED").mean(),
        "node_fail_gpu_hours": g.loc[g.state == "NODE_FAIL", "gpu_hours"].sum(),
    }), include_groups=False).round(3)
by_tier

## Part 3 — Empirical MTBF from NODE_FAIL
The quantity the [checkpoint-interval math](https://sys-design-primer-cvw.pages.dev/google-interview/6-answer-pod-training/) needs: infrastructure failures per GPU-hour, from **large long-running jobs** (≥64 GPUs, ≥1 h — the population that resembles a training run). Two estimators bracket the truth:
- **Lower bound:** `NODE_FAIL` only (clean infra signal, but undercounts — some infra faults end as generic `FAILED`).
- **Upper-ish:** `NODE_FAIL` + a fraction of `FAILED` (the NSDI paper attributes a substantial minority of large-job FAILEDs to infrastructure; 0.3 is a defensible knob — vary it).

In [ ]:
big = gpu_jobs[(gpu_jobs.gpu_num >= 64) & (gpu_jobs.duration >= 3600)]
gh = big.gpu_hours.sum()
nf = (big.state == "NODE_FAIL").sum()
f  = (big.state == "FAILED").sum()
print(f"large jobs: {len(big)},  gpu-hours: {gh:,.0f},  NODE_FAIL: {nf},  FAILED: {f}")
for name, lam in (("NODE_FAIL only", nf/gh), ("NODE_FAIL + 0.3·FAILED", (nf+0.3*f)/gh)):
    print(f"{name:24s} λ={lam:.2e}/GPU-h → per-GPU MTBF {1/lam/8760:.1f} yr → fleet MTBF @17,920 chips: {1/(lam*17920):.1f} h")
print("pod-training prior: 5 yr/chip → fleet MTBF ≈ 2.4 h  (deliberately conservative)")

**You should land near:** per-GPU MTBF ≈ **15–21 years**, fleet MTBF at 17,920 chips ≈ **7–10 h** — i.e. the article's 5-year prior is *pessimistic by ~3–4×* on this cluster's evidence. Interpretation questions:
1. Which way do the job-record biases push this estimate? (One failure = one record even if 3 nodes died → undercount; TPU≠A100; six months of one fleet → wide error bars.)
2. Recompute Young's interval τ* = √(2C·MTBF) with your empirical fleet MTBF and C = 2 s. Does "checkpoint every few minutes" survive a 4× better MTBF? (It should — τ* scales with √MTBF, so a 4× MTBF improvement only doubles the interval.)
3. At 5 s/step, how many steps does one NODE_FAIL event cost with a 15-min restore vs a 2-s-checkpoint + fast restore? Multiply by events/month at your λ.

## Part 4 — The empirical goodput waterfall
GPU-hours by terminal state, plus queue-wait as the scheduling-goodput proxy. **Spoiler to reason about, not around:** CANCELLED dominates GPU-hours (~66%!). That is *not* waste — big pretraining jobs are often run open-ended and cancelled when converged, so a cancelled job's hours were mostly productive. This is the single best example in the dataset of why *terminal state ≠ outcome quality*, and why goodput accounting needs semantics, not just states.

In [ ]:
wf = gpu_jobs.groupby("state").gpu_hours.sum().sort_values(ascending=False)
print((100 * wf / wf.sum()).round(2).to_string())
qw = ((gpu_jobs.start_time - gpu_jobs.submit_time).dt.total_seconds().clip(lower=0) * gpu_jobs.gpu_num).sum() / 3600
print(f"\nqueue-wait GPU-hours as % of consumed: {100*qw/wf.sum():.1f}%   (~3% — scheduling goodput proxy)")
print(f"NODE_FAIL GPU-hours (large jobs): {big.loc[big.state=='NODE_FAIL','gpu_hours'].sum():,.0f} — hours *in* failed jobs, an upper bound on lost work, not the loss itself (a checkpointed job loses only work since its last checkpoint)")

**Final exercise:** write the two-bar waterfall (naive vs engineered) for *this* cluster the way §5 of the pod-training answer does, using your Part-3 λ, a 15-min-restore naive design, and a 2-s async-checkpoint engineered design. Then the honest paragraph: which numbers here are **measured** (λ_NODE_FAIL, queue-wait), which are **proxied** (infra share of FAILED, lost-work-per-event), which are **assumed** (restore times) — the confidence-ledger habit applied to your own analysis.

**Stretch:** the full per-node utilization/power telemetry (~80 GB) is on [HuggingFace](https://huggingface.co/datasets/InternLM/AcmeTrace); with it you can attribute large-job stalls to specific nodes — real straggler forensics at fleet scale.